# 강의자용 완성본 1 — PDF 기반 검색기
이 파일은 **강의자가 먼저 모든 셀을 순서대로 실행할 수 있는 완성 코드**입니다. 학생용 TODO는 포함하지 않습니다.
PDF 선택: [노션 참고자료](https://app.notion.com/p/400b2e6bf807804e8a0ef17344369725).
실습 1에서 문서 한 개를 업로드합니다. 실습 5의 `selected_type`은 선택한 문서 A/B/C에 맞춰 선택합니다.
아래 실습 0부터 차례대로 Shift+Enter를 누릅니다. 각 단계는 앞 단계의 변수들을 사용합니다.
LLM은 선택 사항이며 API 비용이 발생할 수 있습니다.


## 실습 0. 라이브러리 설치
필요 패키지를 설치합니다. 설치가 완료되면 셀 1을 실행합니다.

**교수자 설명:** 입력값, 처리 함수, 출력값을 실행 전에 학생과 예측하고 실제 결과를 비교합니다.


In [ ]:
%pip -q install pypdf scikit-learn



## 실습 1. 선택 PDF 업로드 및 추출
참고자료에서 PDF 한 개를 다운로드해 업로드합니다. 파일명·페이지 수와 텍스트를 확인합니다.

**교수자 설명:** 입력값, 처리 함수, 출력값을 실행 전에 학생과 예측하고 실제 결과를 비교합니다.


In [ ]:
from pathlib import Path
from pypdf import PdfReader

# 실습 1: 제공된 자료 A/B/C 중 하나를 직접 내려받아 Colab에 업로드합니다.
# 반드시 본인이 선택한 PDF 한 개만 현재 작업 파일로 지정합니다.
from google.colab import files
uploaded = files.upload()
pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
assert len(pdf_files) == 1, "PDF 한 개만 업로드하세요."
pdf_name = pdf_files[0]
pdf_path = Path(pdf_name)
assert pdf_path.exists()
print("선택한 PDF:", pdf_name)

reader = PdfReader(str(pdf_path))
records = []
for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()
    if text:
        records.append({"page": page_number, "text": text})

assert records, "추출 텍스트가 없습니다. 스캔 PDF라면 OCR이 필요합니다."
print("텍스트가 있는 페이지:", len(records))\nprint("선택 문서:", pdf_name)



## 실습 2. 청크 분할
PDF 페이지별 텍스트를 나누되 페이지 번호를 보존합니다.

**교수자 설명:** 입력값, 처리 함수, 출력값을 실행 전에 학생과 예측하고 실제 결과를 비교합니다.


In [ ]:
def chunk_words(text, size=75, overlap=20):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("0 <= overlap < size 조건을 확인하세요.")
    words = text.split()
    step = size - overlap
    return [
        " ".join(words[i:i + size])
        for i in range(0, len(words), step)
        if words[i:i + size]
    ]

chunk_size = 75
overlap = 20
chunks = [
    {"page": item["page"], "text": part}
    for item in records
    for part in chunk_words(item["text"], chunk_size, overlap)
]
print("생성 청크:", len(chunks))
print("첫 청크:", chunks[0]["text"][:100])



## 실습 3. 문서 벡터화
문자 2~4그램 TF-IDF로 문서 청크를 벡터화합니다. 의미 임베딩과 다릅니다.

**교수자 설명:** 입력값, 처리 함수, 출력값을 실행 전에 학생과 예측하고 실제 결과를 비교합니다.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4)
)
document_vectors = vectorizer.fit_transform(
    [part["text"] for part in chunks]
)
print("행렬 크기:", document_vectors.shape)



## 실습 4. 질문 검색
질문을 벡터화하고 코사인 유사도 상위 문서 3개와 출처를 확인합니다.

**교수자 설명:** 입력값, 처리 함수, 출력값을 실행 전에 학생과 예측하고 실제 결과를 비교합니다.


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(question, top_k=3):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(
        question_vector, document_vectors
    ).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "page": chunks[int(i)]["page"],
            "text": chunks[int(i)]["text"],
            "score": float(scores[int(i)])
        }
        for i in order
    ]

results = retrieve("문서의 주요 목적은 무엇인가요?", top_k=2)
for item in results:
    print(f"페이지 {item['page']} / 점수 {item['score']:.3f}")
    print(item["text"][:100])



## 실습 5. 문서별 질문 검증
PDF A/B/C에 맞는 질문 세 개를 골라 실제 검색 결과표를 확인합니다.

**교수자 설명:** 입력값, 처리 함수, 출력값을 실행 전에 학생과 예측하고 실제 결과를 비교합니다.


In [ ]:
import pandas as pd

# 자료 1/4: A (사용설명서), 자료 2: B (안전관리), 자료 3: C (연구보고서)
selected_type = "B"  # 시연 PDF와 일치하도록 A/B/C 중 선택
sample_questions = {
    "A": [
        "설명서의 설치 및 시작 절차는 무엇인가요?",
        "주요 기능을 어떻게 사용하나요?",
        "사용 중 주의사항은 무엇인가요?"
    ],
    "B": [
        "안전관리 계획의 목적은 무엇인가요?",
        "안전관리 조직과 담당 업무는 무엇인가요?",
        "사고 발생 시 대응 절차는 무엇인가요?"
    ],
    "C": [
        "연구의 목적은 무엇인가요?",
        "조사 대상과 방법은 무엇인가요?",
        "종합적 지원체계의 개선 방안은 무엇인가요?"
    ]
}
questions = sample_questions[selected_type]
rows = []
for question in questions:
    hit = retrieve(question, top_k=1)[0]
    rows.append({
        "질문": question,
        "페이지": hit["page"],
        "유사도": round(hit["score"], 3)
    })
result_table = pd.DataFrame(rows)
display(result_table)



## 공동 실행 검증 체크리스트
- [ ] 실습 0: 라이브러리 설치
- [ ] 실습 1: PDF 파일명과 추출된 페이지 수 표시
- [ ] 실습 2: 청크 수와 첫 번째 청크 출력
- [ ] 실습 3: TF-IDF 행렬 크기 출력
- [ ] 실습 4: 검색 근거와 점수 출력
- [ ] 실습 5: **선택한 PDF**에 맞는 질문 3개 결과표 출력

각 단계 후 학생용 워크북에서 대응하는 번호의 TODO를 완성하도록 합니다. **실제 Google Colab UI 및 API 유료 호출은 강의 현장에서 추가로 확인합니다.**
